# DNA-aware stochastic model with constraints (AUTO-RUN)

This notebook implements **Option B**:
1) build synthetic data via the mechanistic simulator
2) fit surrogate models (fast predictors)
3) search candidate settings with surrogate constraints
4) verify top candidates by Monte Carlo simulation

### Hard requirements encoded here
- **QC mapping rate ≥ 30%** (i.e., mapping_rate ≥ 0.30)
- **Sequencing reads_total ≥ 3,000,000**
- **Cells plated = 8,000,000**, assume **transfection rate = 70%** → effective transfected cells = 5,600,000


In [ ]:
import os, sys
sys.path.append('.')
sys.path.append('/Users/ds39/PycharmProjects/MAVE_Project/Simulation_Prediction_modelling/26_feb_modelling')
sys.path.append('/mnt/data')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sge_model_skew_dna_mapping_v4 as mod
print('Imported:', mod.__file__)


In [ ]:
# -----------------------------
# SETTINGS (edit these)
# -----------------------------
OUTDIR = '.'  # e.g. './outputs'
os.makedirs(OUTDIR, exist_ok=True)

# Cells + transfection assumption
CELLS_PLATED = 8_000_000
TRANSFECTION_RATE = 0.70
CELLS_TRANSFECTED = mod.effective_transfected_cells(CELLS_PLATED, TRANSFECTION_RATE)
print('Effective transfected cells =', CELLS_TRANSFECTED)

# Sequencing reads
READS_TOTAL = 3_000_000  # minimum required

# Mapping (QC threshold)
MIN_MAPPING_RATE = 0.30
MAPPING_RANGE = (0.30, 0.70)  # used for synthetic data + candidate sampling

# Example DNA amounts
HDR_NG_EXAMPLE = 750
SGRNA_NG_EXAMPLE = 375
RATIO_OPT = 2.0

# Library
LIBRARY_SIZE = 2000
SKEW_SIGMA = 0.5

# Monte Carlo reps
N_REPS_EXAMPLE = 60

# Grid sweep for plots
HDR_VALUES = np.linspace(100, 2000, 9)
SGRNA_VALUES = np.linspace(50, 1500, 9)
N_REPS_GRID = 25

# Synthetic training data
N_SYNTH = 700
N_REPS_PER_SYNTH = 12

# Option B: surrogate search + verify
N_CANDIDATES = 25000
TOP_K_VERIFY = 70
N_REPS_VERIFY = 45

# Targets
TARGET_DROPOUT = 0.02
MIN_P10_READS = 150

print('OUTDIR =', OUTDIR)


In [ ]:
# -----------------------------
# 1) Example Monte Carlo (reports HDR rate + mapping + usable reads)
# -----------------------------
mc = mod.run_monte_carlo(
    n_reps=N_REPS_EXAMPLE,
    hdr_ng=HDR_NG_EXAMPLE,
    sgrna_ng=SGRNA_NG_EXAMPLE,
    ratio_opt=RATIO_OPT,
    cells_transfected=CELLS_TRANSFECTED,
    library_size=LIBRARY_SIZE,
    skew_sigma=SKEW_SIGMA,
    reads_total=READS_TOTAL,
    mapping_rate=MIN_MAPPING_RATE,  # conservative: assume just at QC threshold
)

print('Predicted HDR rate (mean):', mc['hdr_rate_mean'])
print('Mapping rate assumed:', MIN_MAPPING_RATE)
print('Expected usable reads (mean):', mc['reads_usable_mean'])
print('Dropout mean:', mc['dropout_mean'], 'q05..q95:', mc['dropout_q']['0.05'], mc['dropout_q']['0.95'])
print('P10 reads mean:', mc['p10_reads_mean'], 'q05..q95:', mc['p10_reads_q']['0.05'], mc['p10_reads_q']['0.95'])


In [ ]:
# -----------------------------
# 2) Coverage distribution plots (one replicate)
# -----------------------------
rep = mod.simulate_once(
    hdr_ng=HDR_NG_EXAMPLE,
    sgrna_ng=SGRNA_NG_EXAMPLE,
    ratio_opt=RATIO_OPT,
    cells_transfected=CELLS_TRANSFECTED,
    library_size=LIBRARY_SIZE,
    skew_sigma=SKEW_SIGMA,
    reads_total=READS_TOTAL,
    mapping_rate=MIN_MAPPING_RATE,
    return_vectors=True,
)
read_counts = rep['read_counts']

plt.figure()
plt.hist(read_counts, bins=60)
plt.xlabel('Reads per variant')
plt.ylabel('Count of variants')
plt.title('Read-count distribution (one replicate)')
plt.tight_layout()
hist_path = os.path.join(OUTDIR, 'readcount_histogram.png')
plt.savefig(hist_path, dpi=200)
plt.close()

sorted_counts = np.sort(read_counts)
cdf = np.arange(1, len(sorted_counts)+1) / len(sorted_counts)
plt.figure()
plt.plot(sorted_counts, cdf)
plt.xlabel('Reads per variant')
plt.ylabel('CDF')
plt.title('CDF of reads per variant (one replicate)')
plt.grid(True)
plt.tight_layout()
cdf_path = os.path.join(OUTDIR, 'readcount_cdf.png')
plt.savefig(cdf_path, dpi=200)
plt.close()

print('Saved distribution plots:', hist_path, cdf_path)


In [ ]:
# -----------------------------
# 3) Sweep plots with mapping at QC threshold (conservative)
# -----------------------------
dropout_grid = np.zeros((len(HDR_VALUES), len(SGRNA_VALUES)))
p10_grid = np.zeros_like(dropout_grid)

for i, h in enumerate(HDR_VALUES):
    for j, s in enumerate(SGRNA_VALUES):
        mc = mod.run_monte_carlo(
            n_reps=N_REPS_GRID,
            hdr_ng=float(h),
            sgrna_ng=float(s),
            ratio_opt=RATIO_OPT,
            cells_transfected=CELLS_TRANSFECTED,
            library_size=LIBRARY_SIZE,
            skew_sigma=SKEW_SIGMA,
            reads_total=READS_TOTAL,
            mapping_rate=MIN_MAPPING_RATE,
        )
        dropout_grid[i, j] = mc['dropout_mean']
        p10_grid[i, j] = mc['p10_reads_mean']

plt.figure()
plt.imshow(dropout_grid, aspect='auto', origin='lower')
plt.xticks(ticks=np.arange(len(SGRNA_VALUES)), labels=[f'{v:.0f}' for v in SGRNA_VALUES], rotation=45)
plt.yticks(ticks=np.arange(len(HDR_VALUES)), labels=[f'{v:.0f}' for v in HDR_VALUES])
plt.xlabel('sgRNA (ng)')
plt.ylabel('HDR (ng)')
plt.title('Dropout heatmap (HDR x sgRNA) at mapping=0.30')
plt.colorbar(label='Mean dropout')
plt.tight_layout()
heatmap_path = os.path.join(OUTDIR, 'dropout_heatmap_mapping030.png')
plt.savefig(heatmap_path, dpi=200)
plt.close()

print('Saved:', heatmap_path)


In [ ]:
# -----------------------------
# 4) Synthetic training data (mapping_rate >= 0.30) + save
# -----------------------------
rows = mod.build_synthetic_dataset(
    n_samples=N_SYNTH,
    n_reps_per_sample=N_REPS_PER_SYNTH,
    mapping_range=MAPPING_RANGE,
    reads_total=READS_TOTAL,
    cells_transfected=CELLS_TRANSFECTED,
    library_size=LIBRARY_SIZE,
    ratio_opt=RATIO_OPT,
)
df_train = pd.DataFrame(rows)
train_csv = os.path.join(OUTDIR, 'synthetic_training_data_constraints.csv')
df_train.to_csv(train_csv, index=False)
print('Saved synthetic training data:', train_csv)
df_train[['mapping_rate','reads_usable_mean','dropout_mean','p10_reads_mean']].describe()


In [ ]:
# -----------------------------
# 5) Fit surrogates
# -----------------------------
dropout_model, p10_model, feature_names = mod.fit_surrogate_models(rows)
print('Feature names:', feature_names)


In [ ]:
# -----------------------------
# 6) Option B: surrogate search + Monte Carlo verify
# Enforces mapping_rate >= 0.30 and reads_total >= 3,000,000
# -----------------------------
df_verified = mod.suggest_experiments_surrogate_verified(
    dropout_model=dropout_model,
    p10_model=p10_model,
    feature_names=feature_names,
    n_candidates=N_CANDIDATES,
    top_k_to_verify=TOP_K_VERIFY,
    target_dropout=TARGET_DROPOUT,
    min_p10_reads=MIN_P10_READS,
    min_mapping_rate=MIN_MAPPING_RATE,
    min_reads_total=READS_TOTAL,
    mapping_range=MAPPING_RANGE,
    reads_total=READS_TOTAL,
    cells_transfected=CELLS_TRANSFECTED,
    library_size=LIBRARY_SIZE,
    ratio_opt=RATIO_OPT,
    n_reps_verify=N_REPS_VERIFY,
)

out_csv = os.path.join(OUTDIR, 'suggested_experiments_verified_constraints.csv')
df_verified.to_csv(out_csv, index=False)
print('Saved:', out_csv)
df_verified.head(20)


## About mapping values like 0.10–0.30
- Yes: `0.10–0.30` means **10%–30%** mapping rate.
- With QC requiring **≥30%**, this notebook enforces **mapping_rate ≥ 0.30** during synthetic training and candidate search.

## Note on “3M reads”
- Here `READS_TOTAL = 3,000,000` is **raw reads**.
- At mapping=0.30, usable reads are ~900k.
- If you instead mean **3M usable mapped reads**, set `READS_TOTAL = 10,000,000` (because 10M×0.30 = 3M).
